<a href="https://colab.research.google.com/github/akashmavle5/--akash/blob/main/Akash_Panini_Gen_AI001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
panini_engine.py
A rule-based engine inspired by Panini's Ashtadhyayi.
Given a word, it:
  1. Generates similar words (via vowel gradation, sandhi, affixation).
  2. Generates metaphors (via mapping to Paninian concepts).
"""

from __future__ import annotations
import random
from typing import List, Dict, Tuple

# ---------- 2A: Derivational rules (create new concepts) ----------
DERIVATIONAL_RULES: Dict[str, str] = {
    "krt":            "Affix after root → new lexical item",
    "taddhita":       "Affix after stem → derivative",
    "shashthi":       "Genitive relation → new compound meaning",
    "sup_sup":        "Case + stem → new syntactic unit",
    "vrddhi":         "Vowel upgrade → semantic intensification",
    "guna":           "Vowel middle-grade → semantic moderation",
    "samyoga":        "Consonant cluster → new phonetic unit",
    "vrddhad_yan":    "Specific upgrade → specific new form",
    "adyantau_tak":   "Boundary markers delimit a new domain",
    "lit_dhato":      "Perfect-tense affix → completed-action concept",
}

# ---------- 2B: Metaphorical rules ----------
METAPHOR_RULES: Dict[str, str] = {
    "sthanivad_bhava":  "The replacement carries the soul of the original.",
    "tasya_lopah":      "What is removed still shapes what remains.",
    "agama":            "Insertion enriches without destroying.",
    "adesha":           "Substitution preserves function, changes form.",
    "sandhi":           "Junction of two entities births a third.",
    "it_marker":        "Silent guides govern audible behaviour.",
    "karaka":           "Semantic roles mirror cosmic relations.",
    "apavada":          "The singular reveals the limits of the universal.",
    "priority_chain":   "Eternity > interiority > force.",
    "lupta":            "The unexpressed is still meaningful.",
}

# ---------- 2C: Meta-rules (Paribhasha) ----------
METARULES: Dict[str, str] = {
    "apavada_over_utsarga": "Exception overrides the general.",
    "nitya_baliyat":        "Eternal > internal > stronger.",
    "vipratisedhe_para":    "On conflict, the later rule wins.",
    "karyat_sankhya":       "Purpose > number > strength.",
    "sthanivad_inherit":    "Substitute inherits all properties of original.",
    "alpadhikare_utsarga":  "General rule applies in narrow scope.",
    "bahir_antarange":      "External is weaker than internal.",
    "karyantaram":          "Specific purpose overrides general.",
    "samjna_paribhasha":    "Definitions themselves are meta-rules.",
    "paribhasha_parts":     "Meta-rules apply to parts too.",
}

# ---------- Phonology helpers ----------
VRDDHI = {"a": "ai", "ai": "au", "au": "ai", "e": "ai", "o": "au"}
GUNA   = {"a": "a",  "i": "e",  "e": "a",  "u": "o",  "o": "a",
          "ri": "ar", "ai": "a", "au": "a"}
SEMIVOWEL = {"i": "y", "u": "v", "ri": "r", "lri": "l"}
PREFIXES  = ["pra", "upa", "sam", "vi", "ni", "anu", "abhi", "pari"]
SUFFIXES  = ["aka", "ika", "ta", "tva", "maya", "vant", "in", "man"]

VOWELS = set("aeiou")


def _vrddhi(word: str) -> str:
    """Rule 1: vrddhir adaic — upgrade first vowel."""
    out = list(word)
    for i, ch in enumerate(out):
        if ch in VRDDHI:
            out[i] = VRDDHI[ch]
            break
    return "".join(out)


def _guna(word: str) -> str:
    """Rule 2: adengunah — middle-grade first vowel."""
    out = list(word)
    for i, ch in enumerate(out):
        if ch in GUNA:
            out[i] = GUNA[ch]
            break
    return "".join(out)


def _iko_yan(word: str) -> str:
    """Rule 3: iko yanaci — i/u/ri become semivowels before vowels."""
    out = list(word)
    for i, ch in enumerate(out):
        if ch in SEMIVOWEL and i + 1 < len(word) and word[i + 1] in VOWELS:
            out[i] = SEMIVOWEL[ch]
    return "".join(out)


def _lopa(word: str) -> str:
    """Rule 4: tasya lopah — drop the middle vowel."""
    vowels = [i for i, c in enumerate(word) if c in VOWELS]
    if len(vowels) >= 2:
        out = list(word)
        out.pop(vowels[1])
        return "".join(out)
    return word


def _sandhi(a: str, b: str) -> str:
    """Rule 5 (sandhi): junction — last vowel of a + first of b fuse."""
    if not a or not b:
        return a + b
    if a[-1] in VOWELS and b[0] in VOWELS:
        return a[:-1] + a[-1] + b[1:]
    return a + b


def _prefix(word: str) -> str:
    return random.choice(PREFIXES) + word


def _suffix(word: str) -> str:
    return word + random.choice(SUFFIXES)


def _boundary_mark(word: str) -> str:
    """Rule 7: halantyam — tag the word with a silent marker."""
    return word + "'"


def _resonate(word: str) -> str:
    """Rule 8: upadeshe 'janun — nasalise the first vowel."""
    out = list(word)
    for i, ch in enumerate(out):
        if ch in VOWELS:
            out[i] = ch + "n"
            break
    return "".join(out)


def _terminal_align(word: str) -> str:
    """Rule 9: tulo'ntyena — echo the final letter at the start."""
    if word:
        return word[-1] + word
    return word


def _sequential_unify(words: List[str]) -> str:
    """Rule 10: parascaikasminn — fuse a sequence into one unit."""
    return "".join(w[:2] for w in words if w)


# ---------- Similar-word generator ----------
def generate_similar(word: str, n: int = 8) -> List[Tuple[str, str]]:
    """
    Apply each of the 10 derivational rules to produce a 'similar' form,
    each tagged with the rule that produced it.
    """
    w = word.strip().lower()
    transforms = [
        ("vrddhir_adaic",        _vrddhi(w)),
        ("adeng_gunah",          _guna(w)),
        ("iko_yanaci",           _iko_yan(w)),
        ("tasya_lopah",          _lopa(w)),
        ("sandhi_with_self",     _sandhi(w, w)),
        ("prefixation",          _prefix(w)),
        ("suffixation",          _suffix(w)),
        ("hal_antyam_marker",    _boundary_mark(w)),
        ("upadeshe_resonance",   _resonate(w)),
        ("tulo_antyena",         _terminal_align(w)),
    ]
    # de-duplicate while preserving order
    seen, out = set(), []
    for rule, form in transforms:
        if form and form != w and form not in seen:
            seen.add(form)
            out.append((rule, form))
    return out[:n]


# ---------- Metaphor generator ----------
def generate_metaphors(word: str, n: int = 5) -> List[Tuple[str, str, str]]:
    """
    Map the input word to n metaphorical interpretations drawn from
    the 10 metaphorical rules.  Returns (rule, metaphor, gloss).
    """
    w = word.strip()
    rules = list(METAPHOR_RULES.items())
    random.shuffle(rules)
    out = []
    for key, meaning in rules[:n]:
        metaphor = f"'{w}' is like {meaning.lower()}"
        out.append((key, metaphor, meaning))
    return out


# ---------- Interactive driver ----------
def main() -> None:
    print("=" * 60)
    print("  PANINI ENGINE — Ashtadhyayi-inspired word workshop")
    print("=" * 60)
    word = input("\nEnter a word: ").strip()
    if not word:
        print("No word entered. Exiting.")
        return

    print(f"\n>>> Similar words derived from '{word}':")
    for rule, form in generate_similar(word):
        print(f"  [{rule:<22}]  {form}")

    print(f"\n>>> Metaphors for '{word}':")
    for rule, metaphor, gloss in generate_metaphors(word):
        print(f"  [{rule:<20}]  {metaphor}")
        print(f"       gloss: {gloss}")

    print("\n>>> Meta-rules currently active:")
    for k, v in list(METARULES.items())[:5]:
        print(f"  • {k}: {v}")


if __name__ == "__main__":
    main()

  PANINI ENGINE — Ashtadhyayi-inspired word workshop

Enter a word: vidya

>>> Similar words derived from 'vidya':
  [vrddhir_adaic         ]  vidyai
  [adeng_gunah           ]  vedya
  [tasya_lopah           ]  vidy
  [sandhi_with_self      ]  vidyavidya
  [prefixation           ]  pravidya
  [suffixation           ]  vidyaman
  [hal_antyam_marker     ]  vidya'
  [upadeshe_resonance    ]  vindya

>>> Metaphors for 'vidya':
  [priority_chain      ]  'vidya' is like eternity > interiority > force.
       gloss: Eternity > interiority > force.
  [it_marker           ]  'vidya' is like silent guides govern audible behaviour.
       gloss: Silent guides govern audible behaviour.
  [karaka              ]  'vidya' is like semantic roles mirror cosmic relations.
       gloss: Semantic roles mirror cosmic relations.
  [sandhi              ]  'vidya' is like junction of two entities births a third.
       gloss: Junction of two entities births a third.
  [apavada             ]  'vidya' is like 